### Jav-Nairobi (Temporal Equity)

### i. libraries

In [1]:
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
import seaborn as sns
import gtfs_kit as gk

### ii. background

Temporal equity looks at how service availability varies over time, especially during peak vs. off-peak hours, and across different regions or population groups.

The main questions:

- Do all population groups have similar access to transit throughout the day or week?
- Do low-income or peripheral areas get fewer trips per hour?
- Are night or weekend services equally distributed?
- Do peak-hour frequencies differ by income zone?

### iii. data

In [2]:
# this dataframe contains the full data from the coverage eda
ward_full_gdf = pd.read_csv('/home/dataopske/Desktop/jav/data/processed/wards_full_gdf.csv')

# load GTFS data
feed_path = '/home/dataopske/Desktop/jav/data/raw/digitalmatatu/GTFS_FEED_2019.zip'
feed = gk.read_feed(feed_path,dist_units='km')

### iv. Descriptive

#### a. Trips per hour per route

In [3]:
frequencies=feed.frequencies

In [4]:
frequencies.head(1)

,trip_id,start_time,end_time,headway_secs
0,10106110,06:00:00,09:00:00,300


- `Headway` is the time gap between two consecutive vehicles (buses, matatus, trains, etc.) on the same route.
- `Hour` simply means which hour of the day that trip starts — derived from start_time in the GTFS data

In [5]:
# Clean start_time to hour
frequencies['start_time'] = pd.to_timedelta(frequencies['start_time'], errors='coerce')
frequencies['hour'] = frequencies['start_time'].dt.seconds // 3600

# Average headway per route-hour 
freq_hour = (
    frequencies.groupby(['trip_id', 'hour'])
    ['headway_secs'].mean()
    .reset_index()
)

# Convert headway to trips/hour
freq_hour['trips_per_hour'] = 3600 / freq_hour['headway_secs']


`trips_per_hour` is the number of vehicles leaving per hour on a route — a direct measure of how frequent the service is.

In [6]:
freq_hour.head(2)

,trip_id,hour,headway_secs,trips_per_hour
0,10106110,6,300.0,12.0
1,10106110,9,900.0,4.0


So if a ward has an average `trips_per_hour` = 4 at 8 AM, that means on average, `four vehicles` serve that ward per hour; 
that’s decent coverage during that time.

#### b. Trips per ward

In [7]:
stop_times=feed.stop_times

In [8]:
stops_df = feed.stops
stops_df.head(2)

,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station
0,0001RLW,Railways,-1.290884,36.828242,0,<NA>
1,0002KOJ,Koja,-1.281230,36.822596,1,<NA>


In [10]:
from shapely import wkt
import geopandas as gpd

# Step 1: Connect trips to stops
trip_stops = stop_times[['trip_id', 'stop_id']].drop_duplicates()

# Step 2: Add frequency data to each stop
stop_service = trip_stops.merge(
    freq_hour[['trip_id', 'hour', 'trips_per_hour']], 
    on='trip_id',
    how='inner'
)

# Step 3: Aggregate by stop and hour
stop_hour_service = (
    stop_service
    .groupby(['stop_id', 'hour'], as_index=False)
    ['trips_per_hour'].sum()
)

# Step 4: Convert ward geometries from strings to Shapely objects
ward_full_gdf['geometry'] = ward_full_gdf['geometry'].apply(wkt.loads)

# Step 5: Set correct CRS for wards (UTM 37S for Nairobi)
ward_full_gdf = gpd.GeoDataFrame(
    ward_full_gdf,
    geometry='geometry',
    crs="EPSG:32737"
)

# Step 6: Create stops in WGS84 and transform to match wards
stops_gdf = gpd.GeoDataFrame(
    stops_df,
    geometry=gpd.points_from_xy(stops_df.stop_lon, stops_df.stop_lat),
    crs="EPSG:4326"
)
stops_gdf = stops_gdf.to_crs(ward_full_gdf.crs)

# Step 7: Spatial join - assign stops to wards
stops_with_ward = gpd.sjoin(
    stops_gdf,
    ward_full_gdf[['ward', 'population', 'geometry']],
    how='inner',
    predicate='within'
)

# Step 8: Add service frequency to stops
stops_with_service = stops_with_ward.merge(
    stop_hour_service,
    on='stop_id',
    how='inner'
)

# Step 9: Sum service by ward and hour
service_by_ward_hour = (
    stops_with_service
    .groupby(['ward', 'hour'], as_index=False)
    .agg({
        'trips_per_hour': 'sum',
        'population': 'first'
    })
)

# Step 10: Calculate service per capita
service_by_ward_hour['service_per_1k_pop'] = (
    service_by_ward_hour['trips_per_hour'] / 
    service_by_ward_hour['population'] * 1000
)

print(f"Final result: {len(service_by_ward_hour)} ward-hour combinations")
print(service_by_ward_hour.head(10))

Final result: 237 ward-hour combinations
              ward  hour  trips_per_hour     population  service_per_1k_pop
0     Airbase Ward     6          1164.0  105433.255157            11.04016
1     Airbase Ward     9           388.0  105433.255157            3.680053
2     Airbase Ward    15          1164.0  105433.255157            11.04016
3        Babandogo     6           504.0  144280.941622            3.493185
4        Babandogo     9           168.0  144280.941622            1.164395
5        Babandogo    15           504.0  144280.941622            3.493185
6  California Ward     6            36.0   46558.616489            0.773219
7  California Ward     9            12.0   46558.616489             0.25774
8  California Ward    15            36.0   46558.616489            0.773219
9        Clay City     6           732.0   59710.233673           12.259205


In [11]:
# Average service per ward (across all hours)
avg_service_by_ward = service_by_ward_hour.groupby('ward').agg({
    'service_per_1k_pop': 'mean',
    'trips_per_hour': 'sum',
    'population': 'first'
}).reset_index()

In [12]:
avg_service_by_ward.head(1)

,ward,service_per_1k_pop,trips_per_hour,population
0,Airbase Ward,8.586791,2716.0,105433.255157


In [16]:
import folium
from folium import plugins
import json
import pandas as pd
import numpy as np

# 1. AGGREGATED MAP - Average service per ward across all hours

# Aggregate service by ward
avg_service_by_ward = service_by_ward_hour.groupby('ward').agg({
    'service_per_1k_pop': 'mean',
    'trips_per_hour': 'sum',
    'population': 'first'
}).reset_index()

# Ensure numeric values
avg_service_by_ward['service_per_1k_pop'] = pd.to_numeric(avg_service_by_ward['service_per_1k_pop'], errors='coerce')

# Merge back with ward geometries
ward_service_map = ward_full_gdf.merge(
    avg_service_by_ward,
    on='ward',
    how='left'
)

# Transform back to WGS84 for mapping
ward_service_map = ward_service_map.to_crs("EPSG:4326")

# Fill NaN values for mapping
ward_service_map['service_per_1k_pop'] = ward_service_map['service_per_1k_pop'].fillna(0)

# Create the aggregated map
m1 = folium.Map(
    location=[-1.286389, 36.817223],  # Nairobi center
    zoom_start=11,
    tiles='CartoDB positron'
)

# Add choropleth layer
folium.Choropleth(
    geo_data=ward_service_map.to_json(),
    data=ward_service_map,
    columns=['ward', 'service_per_1k_pop'],
    key_on='feature.properties.ward',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Average Transit Service per 1,000 Residents',
    nan_fill_color='lightgray'
).add_to(m1)

# Add tooltips
style_function = lambda x: {
    'fillColor': 'transparent',
    'color': 'transparent',
    'weight': 0
}

highlight_function = lambda x: {
    'weight': 3,
    'color': 'black',
    'fillOpacity': 0.7
}

tooltip = folium.GeoJsonTooltip(
    fields=['ward', 'service_per_1k_pop', 'trips_per_hour', 'population'],
    aliases=['Ward:', 'Service per 1k:', 'Total Trips:', 'Population:'],
    localize=True,
    sticky=False,
    labels=True,
    style="""
        background-color: white;
        border: 2px solid black;
        border-radius: 3px;
        box-shadow: 3px;
    """
)

folium.GeoJson(
    ward_service_map,
    style_function=style_function,
    highlight_function=highlight_function,
    tooltip=tooltip
).add_to(m1)

# Display in notebook
display(m1)

ValueError: autodetected range of [0.0, inf] is not finite

In [17]:
# 2. ANIMATED MAP - Service over time by hour

# Helper function for color
def get_color(value):
    if pd.isna(value) or value == 0:
        return '#d3d3d3'
    elif value < 5:
        return '#fee5d9'
    elif value < 10:
        return '#fcae91'
    elif value < 20:
        return '#fb6a4a'
    elif value < 30:
        return '#de2d26'
    else:
        return '#a50f15'

# Prepare hourly data
ward_hourly = ward_full_gdf.merge(
    service_by_ward_hour,
    on='ward',
    how='left'
)
ward_hourly = ward_hourly.to_crs("EPSG:4326")
ward_hourly['service_per_1k_pop'] = ward_hourly['service_per_1k_pop'].fillna(0)

# Get sorted hours
hours = sorted(ward_hourly['hour'].dropna().unique().astype(int))

# Create timestamped features
features = []
for hour in hours:
    hour_data = ward_hourly[ward_hourly['hour'] == hour]
    
    for idx, row in hour_data.iterrows():
        features.append({
            'type': 'Feature',
            'geometry': json.loads(row['geometry'].to_json()),
            'properties': {
                'times': [f"2024-01-01T{int(hour):02d}:00:00"],
                'style': {
                    'color': 'white',
                    'weight': 1,
                    'fillColor': get_color(row['service_per_1k_pop']),
                    'fillOpacity': 0.7
                },
                'icon': 'circle',
                'iconstyle': {
                    'fillColor': get_color(row['service_per_1k_pop']),
                    'fillOpacity': 0.7,
                    'stroke': 'true',
                    'radius': 5
                },
                'popup': f"<b>{row['ward']}</b><br>Hour: {int(hour):02d}:00<br>Service: {row['service_per_1k_pop']:.1f} per 1k<br>Trips: {row['trips_per_hour']:.0f}"
            }
        })

# Create map
m2 = folium.Map(
    location=[-1.286389, 36.817223],
    zoom_start=11,
    tiles='CartoDB positron'
)

# Add timestamped GeoJSON
plugins.TimestampedGeoJson(
    {
        'type': 'FeatureCollection',
        'features': features
    },
    period='PT1H',
    add_last_point=True,
    auto_play=False,
    loop=True,
    max_speed=2,
    loop_button=True,
    date_options='HH:mm',
    time_slider_drag_update=True
).add_to(m2)

# Add legend
legend_html = '''
<div style="position: fixed; 
     bottom: 50px; right: 50px; width: 200px; height: 180px; 
     background-color: white; border:2px solid grey; z-index:9999; 
     font-size:14px; padding: 10px">
     <p><b>Service per 1k residents</b></p>
     <p><i style="background:#a50f15; width: 20px; height: 20px; display: inline-block;"></i> &gt; 30</p>
     <p><i style="background:#de2d26; width: 20px; height: 20px; display: inline-block;"></i> 20-30</p>
     <p><i style="background:#fb6a4a; width: 20px; height: 20px; display: inline-block;"></i> 10-20</p>
     <p><i style="background:#fcae91; width: 20px; height: 20px; display: inline-block;"></i> 5-10</p>
     <p><i style="background:#fee5d9; width: 20px; height: 20px; display: inline-block;"></i> &lt; 5</p>
     <p><i style="background:#d3d3d3; width: 20px; height: 20px; display: inline-block;"></i> No data</p>
</div>
'''
m2.get_root().html.add_child(folium.Element(legend_html))

# Display in notebook
display(m2)

AttributeError: 'Polygon' object has no attribute 'to_json'